# 02 · LRS-VoxMM data (CPU)
Downloads LRS-VoxMM with your personal KAIST links and makes 96×96 mouth ROIs.

1. Add a Kaggle secret **VOXMM_URLS** containing the download links, one per line (copy them from `chrome://downloads`). They are never printed.
2. Run once with `PART = 'test'`. Then, for the dev set (27k clips), run copies with `PART = 'dev'` and `SHARD = 0, 1, 2`.

Settings: Accelerator **None**, Internet **On**. Use **Save & Run All (Commit)**.

In [ ]:
PART = 'test'             # 'test' or 'dev'
SHARD, NUM_SHARDS = 0, 1   # dev: use NUM_SHARDS = 3

In [ ]:
# --- setup: clone the project code ---
import os, subprocess
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/durgaanuhya02/PVR-Lip-Reading.git", "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
!git log --oneline -1

In [ ]:
# --- load a Kaggle secret into an environment variable (value is never printed) ---
def load_secret(name):
    from kaggle_secrets import UserSecretsClient
    os.environ[name] = UserSecretsClient().get_secret(name)
    print(f"{name} loaded")
load_secret('VOXMM_URLS')

In [ ]:
!pip install -q mediapipe cmudict
!df -h /tmp /kaggle/working; nproc

In [ ]:
!python -m src.preprocess.download_urls --env VOXMM_URLS --out /tmp/lrs_voxmm
!find /tmp/lrs_voxmm -maxdepth 3 -type d | head -20

In [ ]:
import glob
# find the folder of this split: a directory named like the split whose sub-folders hold .mp4 clips
cands = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob('/tmp/lrs_voxmm/**/*.mp4', recursive=True)})
root = [c for c in cands if PART in os.path.basename(c).lower()]
print('candidate roots:', cands)
assert root, f'no folder for {PART}: check the folder list above and set root manually'
root = root[0]; print('using', root)
SPLIT = 'test' if PART == 'test' else 'val'
!python -m src.preprocess.build_lrs_manifest --root {root} --name voxmm/{PART} --split {SPLIT} --out /kaggle/working/voxmm_{PART}_manifest.jsonl
OUT = f'/kaggle/working/voxmm_{PART}_shard{SHARD}'
!python -m src.preprocess.extract_rois --manifest /kaggle/working/voxmm_{PART}_manifest.jsonl --out {OUT} --workers 4 --detect-every 2 --shard {SHARD}/{NUM_SHARDS} 2>&1 | grep -E 'clips|eta' | tail -20
!wc -l {OUT}/index.jsonl {OUT}/errors.jsonl; du -sh {OUT}

In [ ]:
!rm -rf /kaggle/working/repo   # keep only the data in the saved output